# Colab GPU Runner

Thin notebook for the shared Google Drive + local scratch workflow.

- Use `MODE = "primary"` in the first GPU session.
- Use `MODE = "worker"` in the second GPU session.
- Use `MODE = "status"` or `MODE = "finalize"` if you need those commands separately.


In [ ]:
from google.colab import drive

drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import shutil
import socket
import subprocess
import sys
import time

DRIVE_BASE = Path('/content/drive/MyDrive')
REPO_URL = 'https://github.com/andersvestrum/carbon-aware-recsys.git'
REPO_BRANCH = 'big_man_thing'
REPO_ROOT = DRIVE_BASE / 'carbon-aware-recsys'
DRIVE_ROOT = DRIVE_BASE / 'carbon-aware-recsys-colab'
SCRATCH_ROOT = Path('/content/carecs')

MODE = 'primary'  # primary | worker | prepare | status | finalize
WORKER_NAME = 'primary' if MODE == 'primary' else f"worker-{socket.gethostname()}-{int(time.time())}"

CATEGORIES = ['electronics', 'home_and_kitchen', 'sports_and_outdoors']
MODELS = ['BPR', 'NeuMF', 'LightGCN']

TOP_K_CANDIDATES = 100
TOP_K_RERANK = 10
USER_BATCH_SIZE = 1024
MAX_USERS = None
EPOCHS = 50
TRAIN_BATCH_SIZE = 8192
EVAL_BATCH_SIZE = 16384
LEARNING_RATE = 1e-3
EVAL_STEP = 10

def clone_repo():
    if REPO_ROOT.exists():
        print(f'Removing existing directory: {REPO_ROOT}')
        shutil.rmtree(REPO_ROOT)
    subprocess.run([
        'git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, str(REPO_ROOT)
    ], check=True)

if (REPO_ROOT / '.git').exists():
    try:
        subprocess.run(['git', '-C', str(REPO_ROOT), 'fetch', 'origin', REPO_BRANCH], check=True)
        subprocess.run(['git', '-C', str(REPO_ROOT), 'checkout', REPO_BRANCH], check=True)
        subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only', 'origin', REPO_BRANCH], check=True)
    except subprocess.CalledProcessError:
        print('Existing checkout could not switch cleanly; re-cloning the repo.')
        clone_repo()
else:
    clone_repo()

subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(REPO_ROOT / 'requirements.txt')], check=True)
subprocess.run([
    sys.executable,
    '-c',
    'import recbole, kmeans_pytorch, torch; print(recbole.__version__, torch.__version__)',
], check=True)

print({
    'repo_root': str(REPO_ROOT),
    'drive_root': str(DRIVE_ROOT),
    'scratch_root': str(SCRATCH_ROOT),
    'repo_branch': REPO_BRANCH,
    'mode': MODE,
    'worker_name': WORKER_NAME,
})


In [ ]:
import subprocess
import sys

script = REPO_ROOT / 'scripts' / '05_colab_runner.py'
common_args = [
    '--drive-root', str(DRIVE_ROOT),
    '--scratch-root', str(SCRATCH_ROOT),
    '--repo-root', str(REPO_ROOT),
    '--categories', *CATEGORIES,
    '--models', *MODELS,
    '--top-k-candidates', str(TOP_K_CANDIDATES),
    '--top-k-rerank', str(TOP_K_RERANK),
    '--user-batch-size', str(USER_BATCH_SIZE),
    '--epochs', str(EPOCHS),
    '--train-batch-size', str(TRAIN_BATCH_SIZE),
    '--eval-batch-size', str(EVAL_BATCH_SIZE),
    '--learning-rate', str(LEARNING_RATE),
    '--eval-step', str(EVAL_STEP),
]
if MAX_USERS is not None:
    common_args.extend(['--max-users', str(MAX_USERS)])

if MODE == 'primary':
    commands = ['prepare', 'worker', 'finalize']
elif MODE in {'worker', 'prepare', 'status', 'finalize'}:
    commands = [MODE]
else:
    raise ValueError(f'Unsupported MODE: {MODE}')

for command in commands:
    cmd = [sys.executable, str(script), command, *common_args]
    if command in {'worker', 'finalize'}:
        cmd.extend(['--worker-name', WORKER_NAME])
    print('Running:', ' '.join(cmd))
    subprocess.run(cmd, cwd=str(REPO_ROOT), check=True)


In [ ]:
import subprocess
import sys

status_cmd = [
    sys.executable,
    str(REPO_ROOT / 'scripts' / '05_colab_runner.py'),
    'status',
    '--drive-root', str(DRIVE_ROOT),
    '--scratch-root', str(SCRATCH_ROOT),
    '--repo-root', str(REPO_ROOT),
]
subprocess.run(status_cmd, cwd=str(REPO_ROOT), check=True)
